In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Motor de Análisis Retail: Reglas de Asociación Optimizadas (PySpark)
# MAGIC **Arquitectura V2:** Incluye mitigación de explosión cartesiana mediante pre-filtrado Apriori y escritura distribuida nativa.

# COMMAND ----------
# CONFIGURACIÓN DE ENTORNO SEGURO EN COLAB
import sys

if 'google.colab' in sys.modules:
    !pip install pyspark -q
    from google.colab import drive
    drive.mount('/content/drive')

# COMMAND ----------
import os
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, round

# Inicialización de sesión Spark optimizada para manejo de memoria
spark = SparkSession.builder \
    .appName("Retail_Market_Basket_Optimization") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# COMMAND ----------
# MAGIC %md
# MAGIC ### 1. Ingesta de Datos
# COMMAND ----------

if 'google.colab' in sys.modules:
    DB_PATH = '/content/drive/MyDrive/sano y fresco/sanoyfresco.db'
else:
    DB_PATH = 'sanoyfresco.db'

if not os.path.exists(DB_PATH):
    # En producción real, SQLite se reemplazaría por ADLS (Azure Data Lake) o Unity Catalog.
    print(f"Advertencia: Archivo local no encontrado en {DB_PATH}. Usando datos de prueba para validación de clúster.")
    # Fallback seguro para revisión de código si no existe la DB
    data = [(1, "Manzana", 101, 10, 1), (1, "Pera", 102, 10, 1), (2, "Manzana", 101, 10, 1), (2, "Plátano", 103, 10, 1)]
    df_tickets = spark.createDataFrame(data, ["id_pedido", "nombre_producto", "id_producto", "id_seccion", "id_departamento"])
else:
    try:
        # Nota de Arquitectura: La lectura por Pandas carga todo al nodo Driver (Riesgo OOM).
        # En clústeres puros, se conectaría vía JDBC directo a RDBMS o lectura Delta/Parquet.
        conexion = sqlite3.connect(DB_PATH)
        query = "SELECT id_pedido, nombre_producto, id_producto, id_seccion, id_departamento FROM tickets"
        df_tickets_pandas = pd.read_sql_query(query, conexion)
        df_tickets = spark.createDataFrame(df_tickets_pandas)
        print("ÉXITO: Datos integrados al clúster de Spark.")
    except Exception as e:
        print(f"Fallo crítico en la capa de ingesta: {e}")
        raise
    finally:
        if 'conexion' in locals():
            conexion.close()

# COMMAND ----------
# MAGIC %md
# MAGIC ### 2. Optimización Apriori (Poda Previa al Self-Join)
# MAGIC **Intervención Arquitectónica:** Evitamos la explosión cartesiana filtrando primero el catálogo.
# COMMAND ----------

# 1. Calcular base del denominador
total_pedidos = df_tickets.select("id_pedido").distinct().count()

# Umbral base para evitar procesar ruido estadístico (ej. productos que se venden en < 1% de tickets)
UMBRAL_SOPORTE_MINIMO = 0.01

# 2. Calcular soporte individual PRIMERO
df_soporte_individual = df_tickets.groupBy("nombre_producto") \
    .agg((count("id_pedido") / total_pedidos).alias("soporte_individual"))

# 3. Filtrar catálogo para retener solo productos comercialmente relevantes
df_productos_frecuentes = df_soporte_individual.filter(col("soporte_individual") >= UMBRAL_SOPORTE_MINIMO)

# 4. Poda del dataset original: Solo dejamos tickets de productos que sobrevivieron al umbral
# Esto reduce el costo del Self-Join (Shuffle) de O(M^2) a O(K^2) donde K << M
df_tickets_reducidos = df_tickets.join(df_productos_frecuentes, "nombre_producto", "inner")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 3. Construcción de Matriz de Co-ocurrencia (Self-Join Optimizado)
# COMMAND ----------

tgt_df = df_tickets_reducidos.alias("tgt")
src_df = df_tickets_reducidos.alias("src")

join_condition = (col("tgt.id_pedido") == col("src.id_pedido")) & \
                 (col("tgt.nombre_producto") < col("src.nombre_producto"))

df_pares_coocurrencia = tgt_df.join(src_df, join_condition, "inner") \
    .groupBy(
        col("tgt.nombre_producto").alias("antecedente"),
        col("src.nombre_producto").alias("consecuente")
    ) \
    .agg(count("tgt.id_pedido").alias("frecuencia_conjunta"))

# COMMAND ----------
# MAGIC %md
# MAGIC ### 4. Métricas Vectoriales y Capa Gold
# COMMAND ----------

UMBRAL_CONFIANZA = 5.0 # 5%

# Ensamblaje final
results_df = df_pares_coocurrencia \
    .join(df_soporte_individual.alias("sop_ant"), col("antecedente") == col("sop_ant.nombre_producto"), "inner") \
    .join(df_soporte_individual.alias("sop_cons"), col("consecuente") == col("sop_cons.nombre_producto"), "inner") \
    .select(
        col("antecedente"),
        col("consecuente"),
        round((col("sop_ant.soporte_individual") * 100), 2).alias("soporte_a_pct"),
        round(((col("frecuencia_conjunta") / total_pedidos) / col("sop_ant.soporte_individual") * 100), 2).alias("confianza_pct"),
        round(((col("frecuencia_conjunta") / total_pedidos) / (col("sop_ant.soporte_individual") * col("sop_cons.soporte_individual"))), 2).alias("lift")
    ) \
    .filter(col("confianza_pct") > UMBRAL_CONFIANZA) \
    .orderBy(col("lift").desc())

# Metadatos del catálogo
df_productos_maestro = df_tickets.select("nombre_producto", "id_producto", "id_seccion", "id_departamento").distinct()

# Compatibilidad Spark < 3.4 asegurada usando withColumnRenamed en cadena
df_portafolio_final = results_df \
    .join(df_productos_maestro, col("antecedente") == col("nombre_producto"), "left") \
    .drop("nombre_producto") \
    .withColumnRenamed("id_producto", "id_producto_a") \
    .withColumnRenamed("id_seccion", "id_seccion_a") \
    .withColumnRenamed("id_departamento", "id_departamento_a")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 5. Exportación Distribuida de Artefactos
# COMMAND ----------

# Arquitectura Cloud: Se abandona .toPandas() para evitar cuellos de botella en el nodo master.
# coalesce(1) unifica las particiones de Spark en un solo archivo de salida.
output_path = "/tmp/reglas_optimizadas_pyspark" # Cambiar ruta según entorno

df_portafolio_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ";") \
    .csv(output_path)

print(f"ÉXITO: Pipeline finalizado de forma distribuida. Datos guardados en {output_path}")